<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment2/partA_method1/notebook2_partA_method1_birdnet_embeddings_ml_classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Part A Method 1  
## BirdNET embeddings + classical ML classifiers

This notebook uses the BirdNET acoustic embeddings already generated in Notebook 1.

**Input folder**

```text
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean
```

**Expected data**

```text
X_train:        (28357, 1024)
y_train:        (28357,)
meta_train:     (28357, 3)

X_validation:   (7192, 1024)
y_validation:   (7192,)
meta_validation:(7192, 3)
```

**Goal**

Train and compare ML classifiers on BirdNET mean-pooled embeddings:

- Dummy baseline
- Logistic Regression
- Ridge Classifier
- Linear SVM
- SGD Logistic classifier
- Extra Trees classifier

The notebook saves trained models, validation metrics, plots, classification reports, and best-model artefacts.

## 1. Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import json
import time
import warnings
import pickle
import joblib
import re
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Main input path from Notebook 1
# ---------------------------------------------------------------------
EMBEDDINGS_DIR = Path("/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean")

# Output folder for Notebook 2
RESULTS_DIR = EMBEDDINGS_DIR / "notebook2_ml_results"
MODELS_DIR = RESULTS_DIR / "models"
PLOTS_DIR = RESULTS_DIR / "plots"
REPORTS_DIR = RESULTS_DIR / "reports"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"

for p in [RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPORTS_DIR, PREDICTIONS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Embeddings folder:", EMBEDDINGS_DIR)
print("Results folder:   ", RESULTS_DIR)
print("Embeddings folder exists:", EMBEDDINGS_DIR.exists())


## 2. Install/import required packages

In [ ]:
# Usually already available in Colab, but this makes the notebook safer.
!pip install -q scikit-learn joblib seaborn


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
    log_loss
)

from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports complete.")


## 3. Robustly load the saved embeddings

The loader first tries common filenames such as `X_train.npy`, `y_train.npy`, `meta_train.csv`, etc.  
If your actual filenames differ slightly, the fallback pattern search should still find them.

In [ ]:
def find_one_file(base_dir: Path, exact_names=None, patterns=None, required=True, label="file"):
    exact_names = exact_names or []
    patterns = patterns or []

    for name in exact_names:
        p = base_dir / name
        if p.exists():
            return p

    hits = []
    for pat in patterns:
        hits.extend(sorted(base_dir.glob(pat)))

    # Prefer final arrays, not chunk files.
    hits = [h for h in hits if "chunks" not in str(h).lower()]

    if hits:
        if len(hits) > 1:
            print(f"Multiple candidates for {label}; using first:")
            for h in hits[:10]:
                print("  ", h.name)
        return hits[0]

    if required:
        raise FileNotFoundError(
            f"Could not find {label} in {base_dir}. "
            f"Tried exact names={exact_names}, patterns={patterns}"
        )
    return None


def load_array(path: Path):
    if path.suffix.lower() == ".npy":
        return np.load(path, allow_pickle=True)
    if path.suffix.lower() == ".npz":
        data = np.load(path, allow_pickle=True)
        keys = list(data.keys())
        print(f"NPZ keys in {path.name}: {keys}")
        for k in ["arr_0", "X", "x", "embeddings", "features", "y", "labels"]:
            if k in data:
                return data[k]
        return data[keys[0]]
    if path.suffix.lower() in [".pkl", ".pickle"]:
        with open(path, "rb") as f:
            return pickle.load(f)
    raise ValueError(f"Unsupported array file type: {path}")


def load_meta(path: Path):
    if path is None:
        return None
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    if suffix in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    raise ValueError(f"Unsupported metadata file type: {path}")


summary_path = EMBEDDINGS_DIR / "embedding_extraction_summary.json"
if summary_path.exists():
    with open(summary_path, "r") as f:
        summary = json.load(f)
    print("Embedding extraction summary:")
    print(json.dumps(summary, indent=4))
else:
    print("No embedding_extraction_summary.json found. Continuing anyway.")

X_train_path = find_one_file(
    EMBEDDINGS_DIR,
    exact_names=["X_train.npy", "x_train.npy", "train_embeddings.npy", "X_train_all.npy", "X_train_mean.npy"],
    patterns=["*X_train*.npy", "*x_train*.npy", "*train*embedding*.npy", "*train*features*.npy", "*X_train*.npz"],
    label="X_train"
)

y_train_path = find_one_file(
    EMBEDDINGS_DIR,
    exact_names=["y_train.npy", "train_labels.npy", "labels_train.npy", "y_train_all.npy"],
    patterns=["*y_train*.npy", "*train*label*.npy", "*labels_train*.npy", "*y_train*.npz"],
    label="y_train"
)

X_val_path = find_one_file(
    EMBEDDINGS_DIR,
    exact_names=["X_validation.npy", "X_val.npy", "x_validation.npy", "validation_embeddings.npy", "X_validation_all.npy", "X_val_mean.npy"],
    patterns=["*X_validation*.npy", "*X_val*.npy", "*validation*embedding*.npy", "*val*embedding*.npy", "*validation*features*.npy", "*X_validation*.npz", "*X_val*.npz"],
    label="X_validation"
)

y_val_path = find_one_file(
    EMBEDDINGS_DIR,
    exact_names=["y_validation.npy", "y_val.npy", "validation_labels.npy", "labels_validation.npy"],
    patterns=["*y_validation*.npy", "*y_val*.npy", "*validation*label*.npy", "*val*label*.npy", "*y_validation*.npz", "*y_val*.npz"],
    label="y_validation"
)

meta_train_path = find_one_file(
    EMBEDDINGS_DIR,
    exact_names=["meta_train.csv", "train_meta.csv", "metadata_train.csv", "meta_train.parquet"],
    patterns=["*meta*train*.csv", "*train*meta*.csv", "*metadata*train*.csv", "*meta*train*.parquet"],
    required=False,
    label="meta_train"
)

meta_val_path = find_one_file(
    EMBEDDINGS_DIR,
    exact_names=["meta_validation.csv", "meta_val.csv", "validation_meta.csv", "metadata_validation.csv", "meta_validation.parquet"],
    patterns=["*meta*validation*.csv", "*meta*val*.csv", "*validation*meta*.csv", "*metadata*validation*.csv", "*meta*validation*.parquet"],
    required=False,
    label="meta_validation"
)

print("\nResolved files:")
print("X_train:      ", X_train_path)
print("y_train:      ", y_train_path)
print("X_validation: ", X_val_path)
print("y_validation: ", y_val_path)
print("meta_train:   ", meta_train_path)
print("meta_val:     ", meta_val_path)

X_train = load_array(X_train_path)
y_train_raw = load_array(y_train_path)
X_validation = load_array(X_val_path)
y_validation_raw = load_array(y_val_path)

meta_train = load_meta(meta_train_path)
meta_validation = load_meta(meta_val_path)

y_train_raw = np.asarray(y_train_raw).reshape(-1)
y_validation_raw = np.asarray(y_validation_raw).reshape(-1)

print("\nLoaded shapes:")
print("X_train:       ", X_train.shape)
print("y_train:       ", y_train_raw.shape)
print("X_validation:  ", X_validation.shape)
print("y_validation:  ", y_validation_raw.shape)
print("meta_train:    ", None if meta_train is None else meta_train.shape)
print("meta_validation:", None if meta_validation is None else meta_validation.shape)


## 4. Sanity checks and label encoding

In [ ]:
assert X_train.ndim == 2, "X_train must be a 2D matrix"
assert X_validation.ndim == 2, "X_validation must be a 2D matrix"
assert X_train.shape[1] == X_validation.shape[1], "Train and validation embedding dimensions differ"
assert len(X_train) == len(y_train_raw), "X_train and y_train length mismatch"
assert len(X_validation) == len(y_validation_raw), "X_validation and y_validation length mismatch"

X_train = np.asarray(X_train, dtype=np.float32)
X_validation = np.asarray(X_validation, dtype=np.float32)

# Label encoding on train + validation together.
# This is important because validation contains a few classes unseen in train.
all_labels_raw = np.concatenate([y_train_raw.astype(str), y_validation_raw.astype(str)])
label_encoder = LabelEncoder()
label_encoder.fit(all_labels_raw)

y_train = label_encoder.transform(y_train_raw.astype(str))
y_validation = label_encoder.transform(y_validation_raw.astype(str))

class_names = label_encoder.classes_
num_classes_total = len(class_names)

train_class_set = set(y_train)
val_class_set = set(y_validation)
unseen_val_classes = sorted(list(val_class_set - train_class_set))
seen_val_classes = sorted(list(val_class_set & train_class_set))

print("Embedding dimension:", X_train.shape[1])
print("Total classes train+validation:", num_classes_total)
print("Classes in train:", len(train_class_set))
print("Classes in validation:", len(val_class_set))
print("Validation classes unseen in train:", len(unseen_val_classes))

if unseen_val_classes:
    print("\nUnseen validation classes:")
    for idx in unseen_val_classes:
        print(f"  {idx}: {class_names[idx]}")

with open(RESULTS_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

label_summary = {
    "num_classes_total": int(num_classes_total),
    "num_classes_train": int(len(train_class_set)),
    "num_classes_validation": int(len(val_class_set)),
    "num_validation_classes_unseen_in_train": int(len(unseen_val_classes)),
    "unseen_validation_classes": [str(class_names[i]) for i in unseen_val_classes]
}
with open(REPORTS_DIR / "label_summary.json", "w") as f:
    json.dump(label_summary, f, indent=4)

print("\nLabel summary saved:", REPORTS_DIR / "label_summary.json")


## 5. Reporting EDA from embeddings and labels

In [ ]:
train_counts = pd.Series(y_train).value_counts().sort_values(ascending=False)
val_counts = pd.Series(y_validation).value_counts().sort_values(ascending=False)

train_dist = pd.DataFrame({
    "class_id": train_counts.index,
    "class_name": [class_names[i] for i in train_counts.index],
    "train_count": train_counts.values
})

val_dist = pd.DataFrame({
    "class_id": val_counts.index,
    "class_name": [class_names[i] for i in val_counts.index],
    "validation_count": val_counts.values
})

display(train_dist.head(20))
display(val_dist.head(20))

train_dist.to_csv(REPORTS_DIR / "train_class_distribution.csv", index=False)
val_dist.to_csv(REPORTS_DIR / "validation_class_distribution.csv", index=False)

print("Saved class distribution CSV files.")


In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(data=train_dist.head(20), x="class_name", y="train_count")
plt.title("Top 20 training classes by sample count")
plt.xlabel("Class")
plt.ylabel("Training samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "top20_train_classes.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(train_dist["train_count"], bins=50, edgecolor="black")
plt.title("Distribution of training samples per class")
plt.xlabel("Samples per class")
plt.ylabel("Number of classes")
plt.yscale("log")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "train_samples_per_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
embedding_stats = {
    "X_train_shape": list(X_train.shape),
    "X_validation_shape": list(X_validation.shape),
    "X_train_nan_count": int(np.isnan(X_train).sum()),
    "X_validation_nan_count": int(np.isnan(X_validation).sum()),
    "X_train_inf_count": int(np.isinf(X_train).sum()),
    "X_validation_inf_count": int(np.isinf(X_validation).sum()),
    "X_train_mean": float(np.mean(X_train)),
    "X_train_std": float(np.std(X_train)),
    "X_validation_mean": float(np.mean(X_validation)),
    "X_validation_std": float(np.std(X_validation))
}

print(json.dumps(embedding_stats, indent=4))

with open(REPORTS_DIR / "embedding_sanity_stats.json", "w") as f:
    json.dump(embedding_stats, f, indent=4)

# Defensive cleanup if any rare numerical issue exists.
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_validation = np.nan_to_num(X_validation, nan=0.0, posinf=0.0, neginf=0.0)


## 6. Metric and evaluation helper functions

Primary metric recommendation for the report:

- **Macro F1**: important because many bird classes are rare.
- **Weighted F1 / Accuracy**: useful but can be dominated by frequent classes.
- **Balanced accuracy**: useful under class imbalance.

Note: validation classes unseen in training cannot be predicted by the ML classifiers. The notebook reports them explicitly.

In [ ]:
BEST_MODEL_SELECTION_METRIC = "f1_macro"  # alternatives: "accuracy", "f1_weighted", "balanced_accuracy"


def safe_model_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", name).strip("_").lower()


def get_estimator_classes(model):
    if hasattr(model, "classes_"):
        return np.asarray(model.classes_)
    if hasattr(model, "named_steps"):
        last_step = list(model.named_steps.values())[-1]
        if hasattr(last_step, "classes_"):
            return np.asarray(last_step.classes_)
    return None


def align_proba_to_all_classes(proba_raw, estimator_classes, all_class_count):
    """Align predict_proba output columns to the global label encoder class indices."""
    full = np.zeros((proba_raw.shape[0], all_class_count), dtype=np.float64)
    if estimator_classes is None:
        return None
    for col, cls_idx in enumerate(estimator_classes):
        full[:, int(cls_idx)] = proba_raw[:, col]
    full = np.clip(full, 1e-12, 1.0)
    full = full / full.sum(axis=1, keepdims=True)
    return full


def compute_top_k(y_true, proba_full, k=3):
    if proba_full is None:
        return np.nan
    try:
        return top_k_accuracy_score(
            y_true,
            proba_full,
            k=k,
            labels=np.arange(proba_full.shape[1])
        )
    except Exception as e:
        print(f"Could not compute top-{k} accuracy:", e)
        return np.nan


def evaluate_model(model, model_name, X_val, y_val, fit_seconds=None):
    start_pred = time.time()
    y_pred = model.predict(X_val)
    predict_seconds = time.time() - start_pred

    proba_full = None
    val_log_loss = np.nan

    if hasattr(model, "predict_proba"):
        try:
            proba_raw = model.predict_proba(X_val)
            estimator_classes = get_estimator_classes(model)
            proba_full = align_proba_to_all_classes(proba_raw, estimator_classes, num_classes_total)
            val_log_loss = log_loss(
                y_val,
                proba_full,
                labels=np.arange(num_classes_total)
            )
        except Exception as e:
            print(f"{model_name}: predict_proba/log_loss not available or failed:", e)

    metrics = {
        "model": model_name,
        "fit_seconds": np.nan if fit_seconds is None else float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "accuracy": float(accuracy_score(y_val, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, y_pred)),
        "f1_macro": float(f1_score(y_val, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_val, y_pred, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(y_val, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_val, y_pred, average="macro", zero_division=0)),
        "top3_accuracy": float(compute_top_k(y_val, proba_full, k=3)) if proba_full is not None else np.nan,
        "top5_accuracy": float(compute_top_k(y_val, proba_full, k=5)) if proba_full is not None else np.nan,
        "log_loss": float(val_log_loss) if np.isfinite(val_log_loss) else np.nan,
    }

    return metrics, y_pred, proba_full


def save_classification_report(model_name, y_true, y_pred):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=np.arange(num_classes_total),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()
    out = REPORTS_DIR / f"classification_report_{safe_model_name(model_name)}.csv"
    report_df.to_csv(out)
    return out, report_df


## 7. Configure ML classifiers

The default list is chosen to be useful but not excessive.

You can turn slow models on/off using the flags below.

In [ ]:
# ---------------------------------------------------------------------
# Runtime flags
# ---------------------------------------------------------------------
RUN_EXTRA_TREES = True       # Usually useful, but slower than linear models
RUN_SGD_LOGISTIC = True      # Fast approximate logistic classifier
RUN_MLP = False              # Optional; can blur boundary with neural Method 2, so default False
SAVE_EACH_MODEL = True

# ---------------------------------------------------------------------
# Model definitions
# ---------------------------------------------------------------------
models = []

models.append((
    "Dummy most frequent",
    DummyClassifier(strategy="most_frequent")
))

models.append((
    "Logistic Regression unbalanced",
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            solver="saga",
            penalty="l2",
            max_iter=1200,
            tol=1e-3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=0
        ))
    ])
))

models.append((
    "Logistic Regression balanced",
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            solver="saga",
            penalty="l2",
            class_weight="balanced",
            max_iter=1200,
            tol=1e-3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=0
        ))
    ])
))

models.append((
    "Ridge Classifier balanced",
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RidgeClassifier(
            alpha=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
))

models.append((
    "Linear SVM balanced",
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=6000,
            random_state=RANDOM_STATE
        ))
    ])
))

if RUN_SGD_LOGISTIC:
    models.append((
        "SGD Logistic balanced",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SGDClassifier(
                loss="log_loss",
                penalty="elasticnet",
                alpha=1e-4,
                l1_ratio=0.05,
                class_weight="balanced",
                max_iter=1000,
                tol=1e-3,
                early_stopping=True,
                validation_fraction=0.1,
                n_jobs=-1,
                random_state=RANDOM_STATE
            ))
        ])
    ))

if RUN_EXTRA_TREES:
    models.append((
        "Extra Trees balanced",
        ExtraTreesClassifier(
            n_estimators=350,
            max_features="sqrt",
            min_samples_leaf=1,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=0
        )
    ))

if RUN_MLP:
    from sklearn.neural_network import MLPClassifier
    models.append((
        "MLP on embeddings",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(512,),
                activation="relu",
                alpha=1e-4,
                batch_size=256,
                learning_rate_init=1e-3,
                max_iter=80,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=RANDOM_STATE,
                verbose=True
            ))
        ])
    ))

print("Models to train:")
for name, _ in models:
    print(" -", name)


In [ ]:
# ---------------------------------------------------------------------
# Resume / skip settings
# ---------------------------------------------------------------------
SKIP_COMPLETED_MODELS = True

# Add exact model names here only if you want to force rerun later.
# Example: FORCE_RERUN_MODELS = ["Logistic Regression balanced"]
FORCE_RERUN_MODELS = []

RESULTS_SO_FAR_PATH = RESULTS_DIR / "model_comparison_results_so_far.csv"
RESULTS_FINAL_PATH = RESULTS_DIR / "model_comparison_results.csv"


def get_model_artifact_paths(model_name):
    safe = safe_model_name(model_name)
    return {
        "model": MODELS_DIR / f"{safe}.joblib",
        "predictions": PREDICTIONS_DIR / f"predictions_{safe}.csv",
        "report": REPORTS_DIR / f"classification_report_{safe}.csv",
    }


def load_existing_results():
    result_frames = []

    if RESULTS_FINAL_PATH.exists():
        result_frames.append(pd.read_csv(RESULTS_FINAL_PATH))

    if RESULTS_SO_FAR_PATH.exists():
        result_frames.append(pd.read_csv(RESULTS_SO_FAR_PATH))

    if not result_frames:
        return pd.DataFrame()

    df_existing = pd.concat(result_frames, ignore_index=True)

    if "model" in df_existing.columns:
        df_existing = df_existing.drop_duplicates(subset=["model"], keep="last")

    return df_existing


def is_model_complete(model_name, existing_results_df):
    if not SKIP_COMPLETED_MODELS:
        return False

    if model_name in FORCE_RERUN_MODELS:
        return False

    paths = get_model_artifact_paths(model_name)

    required_files_exist = all(path.exists() for path in paths.values())

    has_metrics = (
        not existing_results_df.empty
        and "model" in existing_results_df.columns
        and model_name in existing_results_df["model"].values
    )

    return required_files_exist and has_metrics


existing_results_df = load_existing_results()

print("Resume mode enabled:", SKIP_COMPLETED_MODELS)
print("Existing completed metric rows:", 0 if existing_results_df.empty else len(existing_results_df))

if not existing_results_df.empty:
    display(existing_results_df)

## 8. Train and evaluate all classifiers

In [ ]:
all_results = []

# Load existing completed results first
existing_results_df = load_existing_results()

if not existing_results_df.empty:
    all_results.extend(existing_results_df.to_dict(orient="records"))

best_model = None
best_model_name = None
best_score = -np.inf
best_predictions = None
best_proba = None


def update_best_from_metrics(model_name, metrics, model=None, y_pred=None, proba_full=None):
    global best_model, best_model_name, best_score, best_predictions, best_proba

    score = (
        metrics.get("f1_macro", 0.0)
        + 0.1 * metrics.get("f1_weighted", 0.0)
        + 0.05 * metrics.get("accuracy", 0.0)
    )

    if score > best_score:
        best_score = score
        best_model_name = model_name
        best_model = model
        best_predictions = y_pred
        best_proba = proba_full


# First initialise best model from existing results, if available
for row in all_results:
    if "model" in row:
        update_best_from_metrics(row["model"], row)


for model_name, model in models:
    print("=" * 90)
    print(f"Model: {model_name}")
    print("=" * 90)

    paths = get_model_artifact_paths(model_name)

    if is_model_complete(model_name, existing_results_df):
        print("✓ Already completed. Skipping training.")

        # Try to load skipped model only if it becomes useful as the final best model later.
        existing_row = existing_results_df[existing_results_df["model"] == model_name].iloc[-1].to_dict()
        update_best_from_metrics(model_name, existing_row)

        continue

    print("Training required: missing model, predictions, report, metrics, or force-rerun requested.")

    start_fit = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - start_fit

    metrics, y_pred, proba_full = evaluate_model(
        model=model,
        model_name=model_name,
        X_val=X_validation,
        y_val=y_validation,
        fit_seconds=fit_seconds
    )

    # Remove old result row for this model if rerunning
    all_results = [r for r in all_results if r.get("model") != model_name]
    all_results.append(metrics)

    print("\nValidation metrics:")
    for k, v in metrics.items():
        if k == "model":
            continue
        if isinstance(v, float):
            print(f"  {k:20s}: {v:.5f}")
        else:
            print(f"  {k:20s}: {v}")

    # Save classification report
    report_path, _ = save_classification_report(model_name, y_validation, y_pred)
    print("Saved classification report:", report_path)

    # Save predictions
    pred_df = pd.DataFrame({
        "y_true_encoded": y_validation,
        "y_pred_encoded": y_pred,
        "y_true_label": [class_names[i] for i in y_validation],
        "y_pred_label": [class_names[i] for i in y_pred],
        "correct": y_validation == y_pred
    })

    if meta_validation is not None:
        meta_copy = meta_validation.reset_index(drop=True).copy()
        pred_df = pd.concat([meta_copy, pred_df], axis=1)

    pred_path = paths["predictions"]
    pred_df.to_csv(pred_path, index=False)
    print("Saved predictions:", pred_path)

    # Save fitted model
    model_path = paths["model"]
    joblib.dump(model, model_path, compress=3)
    print("Saved fitted model:", model_path)

    # Save interim result table after every completed model
    results_df = pd.DataFrame(all_results).drop_duplicates(
        subset=["model"],
        keep="last"
    ).sort_values(
        by=["f1_macro", "f1_weighted", "accuracy"],
        ascending=False
    )

    results_df.to_csv(RESULTS_SO_FAR_PATH, index=False)
    existing_results_df = load_existing_results()

    update_best_from_metrics(model_name, metrics, model, y_pred, proba_full)

    print(f"Current best model: {best_model_name}")
    print()

## 9. Compare model results

In [ ]:
results_df = pd.DataFrame(all_results).sort_values(
    by=[BEST_MODEL_SELECTION_METRIC, "f1_weighted", "accuracy"],
    ascending=False
).reset_index(drop=True)

display(results_df)

results_path = RESULTS_DIR / "model_comparison_results.csv"
results_df.to_csv(results_path, index=False)

with open(REPORTS_DIR / "best_model_name.txt", "w") as f:
    f.write(str(best_model_name))

print("Saved final comparison:", results_path)
print("Best model:", best_model_name)


In [ ]:
plot_df = results_df.copy()
metric_cols = ["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted"]

plt.figure(figsize=(14, 7))
plot_long = plot_df.melt(id_vars="model", value_vars=metric_cols, var_name="metric", value_name="score")
sns.barplot(data=plot_long, x="model", y="score", hue="metric")
plt.title("Validation metric comparison across ML classifiers")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=35, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model_metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Save best model bundle

In [ ]:
# If the best model was skipped during resume, load it from disk now.
if best_model is None and best_model_name is not None:
    best_model_path_resume = get_model_artifact_paths(best_model_name)["model"]
    best_model = joblib.load(best_model_path_resume)
    print("Loaded best skipped model from:", best_model_path_resume)

if best_predictions is None and best_model_name is not None:
    best_pred_path_resume = get_model_artifact_paths(best_model_name)["predictions"]
    pred_df_resume = pd.read_csv(best_pred_path_resume)
    best_predictions = pred_df_resume["y_pred_encoded"].to_numpy()
    print("Loaded best skipped predictions from:", best_pred_path_resume)

best_bundle = {
    "model_name": best_model_name,
    "selection_metric": BEST_MODEL_SELECTION_METRIC,
    "model": best_model,
    "label_encoder": label_encoder,
    "class_names": class_names,
    "embedding_type": "BirdNET acoustic embeddings",
    "pooling": "mean",
    "embedding_dim": int(X_train.shape[1]),
    "embeddings_dir": str(EMBEDDINGS_DIR),
    "results": results_df.to_dict(orient="records"),
    "label_summary": label_summary,
}

best_model_path = MODELS_DIR / "best_partA_method1_birdnet_embedding_ml_classifier.joblib"
joblib.dump(best_bundle, best_model_path, compress=3)

print("Saved best model bundle:")
print(best_model_path)


## 11. Best-model detailed evaluation

In [ ]:
print("Best model:", best_model_name)

best_report_path, best_report_df = save_classification_report(
    best_model_name + "_BEST",
    y_validation,
    best_predictions
)

print("Best model classification report saved:", best_report_path)

summary_rows = ["accuracy", "macro avg", "weighted avg"]
display(best_report_df.loc[[r for r in summary_rows if r in best_report_df.index]])

top20_val = val_dist.head(20).copy()
top20_rows = []

for _, row in top20_val.iterrows():
    cls_idx = int(row["class_id"])
    mask = y_validation == cls_idx
    total = int(mask.sum())
    correct = int((best_predictions[mask] == cls_idx).sum())
    acc = correct / total if total > 0 else 0.0
    top20_rows.append({
        "class_id": cls_idx,
        "class_name": class_names[cls_idx],
        "validation_count": total,
        "correct": correct,
        "accuracy": acc
    })

top20_perf = pd.DataFrame(top20_rows)
display(top20_perf)

top20_perf.to_csv(REPORTS_DIR / "best_model_top20_validation_class_performance.csv", index=False)

plt.figure(figsize=(14, 6))
sns.barplot(data=top20_perf, x="class_name", y="accuracy")
plt.title(f"Best model accuracy on top 20 validation classes: {best_model_name}")
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "best_model_top20_class_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. Confusion matrix for top validation classes

This version includes an **Other predicted class** column so mistakes outside the top-N classes are not silently hidden.

In [ ]:
TOP_N_CM = 15

top_indices = [int(x) for x in val_dist.head(TOP_N_CM)["class_id"].tolist()]
other_id = -1
labels_for_cm = top_indices + [other_id]
names_for_cm = [class_names[i] for i in top_indices] + ["Other predicted"]

mask_true_top = np.isin(y_validation, top_indices)
y_true_top = y_validation[mask_true_top]
y_pred_top = best_predictions[mask_true_top]

y_pred_mapped = np.array([p if p in top_indices else other_id for p in y_pred_top])

cm = confusion_matrix(y_true_top, y_pred_mapped, labels=labels_for_cm)

plt.figure(figsize=(14, 11))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=names_for_cm,
    yticklabels=[class_names[i] for i in top_indices] + ["Other true"]
)
plt.title(f"Confusion matrix for top {TOP_N_CM} validation classes — {best_model_name}")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "best_model_confusion_matrix_top_classes.png", dpi=150, bbox_inches="tight")
plt.show()


## 13. Unseen validation classes analysis

These validation classes were not present in training, so no classical ML classifier can learn them from the provided training labels.  
This should be mentioned in the report.

In [ ]:
if unseen_val_classes:
    unseen_rows = []
    for cls_idx in unseen_val_classes:
        mask = y_validation == cls_idx
        unseen_rows.append({
            "class_id": int(cls_idx),
            "class_name": class_names[cls_idx],
            "validation_samples": int(mask.sum()),
            "correct_predictions": int((best_predictions[mask] == cls_idx).sum())
        })
    unseen_df = pd.DataFrame(unseen_rows)
    display(unseen_df)
    unseen_df.to_csv(REPORTS_DIR / "unseen_validation_classes_performance.csv", index=False)
else:
    print("No unseen validation classes found.")


## 14. Optional: small hyperparameter search for Logistic Regression

Run this only after the main comparison if Logistic Regression is competitive.  
It trains multiple models, so it can take additional time.

In [ ]:
RUN_LOGREG_C_SEARCH = False

if RUN_LOGREG_C_SEARCH:
    c_values = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
    search_results = []

    for C in c_values:
        name = f"Logistic Regression balanced C={C}"
        print("=" * 80)
        print(name)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=C,
                solver="saga",
                penalty="l2",
                class_weight="balanced",
                max_iter=1500,
                tol=1e-3,
                n_jobs=-1,
                random_state=RANDOM_STATE
            ))
        ])

        t0 = time.time()
        model.fit(X_train, y_train)
        fit_seconds = time.time() - t0

        metrics, y_pred, proba_full = evaluate_model(model, name, X_validation, y_validation, fit_seconds)
        search_results.append(metrics)
        print(metrics)

        joblib.dump(model, MODELS_DIR / f"{safe_model_name(name)}.joblib", compress=3)

    search_df = pd.DataFrame(search_results).sort_values(
        by=[BEST_MODEL_SELECTION_METRIC, "f1_weighted", "accuracy"],
        ascending=False
    )
    display(search_df)
    search_df.to_csv(RESULTS_DIR / "logistic_regression_C_search_results.csv", index=False)
else:
    print("Skipping Logistic Regression C search. Set RUN_LOGREG_C_SEARCH = True to run it.")


## 15. Optional: load the saved best model later

In [ ]:
loaded_bundle = joblib.load(best_model_path)

print("Loaded bundle model name:", loaded_bundle["model_name"])
print("Embedding dimension expected:", loaded_bundle["embedding_dim"])

loaded_model = loaded_bundle["model"]
example_pred = loaded_model.predict(X_validation[:5])
example_labels = loaded_bundle["label_encoder"].inverse_transform(example_pred)

print("Example predictions:")
for i, label in enumerate(example_labels):
    print(i, label)


## 16. Final report notes template

Use these points in the written report:

1. BirdNET embeddings were generated separately using mean pooling.
2. The final embedding matrices were:
   - training: 28,357 × 1,024
   - validation: 7,192 × 1,024
3. Classical ML classifiers were trained directly on the fixed BirdNET embeddings.
4. Metrics compared: accuracy, balanced accuracy, macro F1, weighted F1, and where available top-k accuracy.
5. Macro F1 is important because the dataset is highly imbalanced.
6. Validation contains some classes that are absent from training; those classes are impossible for the ML classifier to learn and should be discussed as a limitation.
7. The best-performing classifier and saved artefacts are available in the `notebook2_ml_results` folder.